In [1]:
import pandas as pd
import numpy as np
import json
import ta
from datetime import datetime, timezone, timedelta

from lib.data import get_cached_data
from lib.analysis import (
    detect_candlestick_patterns,
    find_support_resistance,
    analyze_market_structure,
    build_price_action_signal,
)
from lib.ai import analyze_with_ai

In [2]:
# Higher timeframe data (daily)
print("=" * 60)
print("📊 DAILY TIMEFRAME (1mo de données)")
print("=" * 60)
df_daily = get_cached_data("EURUSD=X", interval="1d", period="1mo")

# Higher timeframe data (1h - proxy pour 4h)
# Note: yfinance/OpenBB ne supporte pas '4h', on utilise '1h' à la place
print("\n" + "=" * 60)
print("📊 1H TIMEFRAME (1mo de données)")
print("=" * 60)
df_1h = get_cached_data("EURUSD=X", interval="1h", period="1mo")

📊 DAILY TIMEFRAME (1mo de données)
✓ Using cached data (age: 8m 32s)

📊 1H TIMEFRAME (1mo de données)
✓ Using cached data (age: 8m 32s)


In [3]:
# Analyse daily
daily_levels = find_support_resistance(df_daily)
daily_structure = analyze_market_structure(df_daily)

# Indicateurs techniques daily
df_daily["rsi"] = ta.momentum.RSIIndicator(close=df_daily["Close"], window=14).rsi()
macd = ta.trend.MACD(close=df_daily["Close"])
df_daily["macd"] = macd.macd()
df_daily["macd_signal"] = macd.macd_signal()
df_daily["ma20"] = df_daily["Close"].rolling(20).mean()

last_daily = df_daily.iloc[-1]
daily_rsi = round(float(last_daily["rsi"]), 1)
daily_macd = "bullish" if last_daily["macd"] > last_daily["macd_signal"] else "bearish"
daily_trend = "uptrend" if float(last_daily["Close"]) > float(last_daily["ma20"]) else "downtrend"

print(f"   RSI:      {daily_rsi}")
print(f"   MACD:     {daily_macd.upper()}")
print(f"   Trend:    {daily_trend.upper()}")
print()

print("📈 Daily - Structure de marché:")
print(f"   Trend: {daily_structure['structure']}")
print(f"   Bias: {daily_structure['bias']}")
print(f"   Range: {daily_structure['price_range_pct']:.2f}%")
print()

print("🎯 Daily - Niveaux clés:")
for lvl in daily_levels:
    emoji = "🟢" if lvl['type'] == 'Support' else "🔴"
    print(f"   {emoji} {lvl['type']}: {lvl['level']:.5f} ({lvl['strength']}, {lvl['touches']} touches)")

# Analyse 1h
levels_1h = find_support_resistance(df_1h)
structure_1h = analyze_market_structure(df_1h)

print("\n" + "─" * 50)
print("📈 1H - Structure de marché:")
print(f"   Trend: {structure_1h['structure']}")
print(f"   Bias: {structure_1h['bias']}")
print(f"   Range: {structure_1h['price_range_pct']:.2f}%")
print()

print("🎯 1H - Niveaux clés:")
for lvl in levels_1h:
    emoji = "🟢" if lvl['type'] == 'Support' else "🔴"
    print(f"   {emoji} {lvl['type']}: {lvl['level']:.5f} ({lvl['strength']}, {lvl['touches']} touches)")

   RSI:      68.0
   MACD:     BEARISH
   Trend:    UPTREND

📈 Daily - Structure de marché:
   Trend: Consolidation (Ranging)
   Bias: Neutral
   Range: 1.70%

🎯 Daily - Niveaux clés:
   🔴 Resistance: 1.14705 (Medium, 2 touches)
   🟢 Support: 1.13545 (Weak, 1 touches)
   🟢 Support: 1.13788 (Weak, 1 touches)

──────────────────────────────────────────────────
📈 1H - Structure de marché:
   Trend: Consolidation (Ranging)
   Bias: Neutral
   Range: 0.77%

🎯 1H - Niveaux clés:
   🟢 Support: 1.13936 (Strong, 10 touches)
   🔴 Resistance: 1.14558 (Strong, 9 touches)
   🔴 Resistance: 1.14338 (Strong, 4 touches)
   🔴 Resistance: 1.14058 (Medium, 2 touches)
   🟢 Support: 1.13682 (Medium, 2 touches)


In [4]:
# Données intraday (5m) pour le signal
print("📊 INTRADAY TIMEFRAME (5m)")
print("=" * 60)
df_5m = get_cached_data("EURUSD=X", interval="5m", period="1d")

# Patterns, niveaux et structure
patterns_5m = detect_candlestick_patterns(df_5m)
levels_5m = find_support_resistance(df_5m)
structure_5m = analyze_market_structure(df_5m)

# Signal de base
signal = build_price_action_signal(df_5m, patterns_5m, levels_5m, structure_5m)

recent_patterns = [p for p in patterns_5m if p['index'] >= len(df_5m) - 5]
if recent_patterns:
    for p in recent_patterns:
        print(f"Pattern:  {p['pattern']} ({p['signal']}) - {p['strength']}")
else:
    print("Aucun pattern candlestick recent")
print()


📊 INTRADAY TIMEFRAME (5m)
✓ Using cached data (age: 8m 32s)
Aucun pattern candlestick recent



In [5]:
# Enrichir le signal avec le contexte des plus hautes timeframes
signal['htf_daily_bias'] = daily_structure['bias']
signal['htf_daily_structure'] = daily_structure['structure']
signal['htf_1h_bias'] = structure_1h['bias']
signal['htf_1h_structure'] = structure_1h['structure']

# Ajouter les indicateurs daily
signal['daily_rsi'] = daily_rsi
signal['daily_macd'] = daily_macd
signal['daily_trend'] = daily_trend

# Ajouter les niveaux daily et 1h comme contexte
signal['daily_support'] = [l['level'] for l in daily_levels if l['type'] == 'Support'][:2]
signal['daily_resistance'] = [l['level'] for l in daily_levels if l['type'] == 'Resistance'][:2]
signal['hourly_support'] = [l['level'] for l in levels_1h if l['type'] == 'Support'][:2]
signal['hourly_resistance'] = [l['level'] for l in levels_1h if l['type'] == 'Resistance'][:2]

print("\n📊 SIGNAL COMBINÉ (Multi-Timeframe)")
print("=" * 70)
print(f"Daily bias:    {daily_structure['bias']:>8}  |  1H bias:      {structure_1h['bias']:>8}")
print(f"Intraday bias: {structure_5m['bias']:>8}  |  Intraday price: {signal['price']:.5f}")

# Vérifier la concordance des timeframes
biases = [daily_structure['bias'], structure_1h['bias'], structure_5m['bias']]
if all(b == 'Bullish' for b in biases):
    print("\n✅ TOUS LES TIMEFRAMES SONT BULLISH - Signal haussier fort")
elif all(b == 'Bearish' for b in biases):
    print("\n✅ TOUS LES TIMEFRAMES SONT BEARISH - Signal baissier fort")
elif daily_structure['bias'] == structure_1h['bias'] == 'Bullish' and structure_5m['bias'] != 'Bullish':
    print("\n⚠️  HTF bullish mais intraday en contretendance - Attendre confirmation")
elif daily_structure['bias'] == structure_1h['bias'] == 'Bearish' and structure_5m['bias'] != 'Bearish':
    print("\n⚠️  HTF bearish mais intraday en contretendance - Attendre confirmation")
else:
    print("\n🔄 Timeframes en désaccord - Prudence recommandée")


📊 SIGNAL COMBINÉ (Multi-Timeframe)
Daily bias:     Neutral  |  1H bias:       Neutral
Intraday bias:  Neutral  |  Intraday price: 1.15274

🔄 Timeframes en désaccord - Prudence recommandée


In [6]:
# Afficher les parametres envoyes a l'IA
print("PARAMETRES ENVOYES A L'IA :")
print("-" * 70)
for key, value in signal.items():
    print(f"  {key:<24} : {value}")
print("-" * 70)


PARAMETRES ENVOYES A L'IA :
----------------------------------------------------------------------
  pair                     : EUR/USD
  price                    : 1.1527377367019653
  session                  : New York
  market_structure         : Consolidation (Ranging)
  bias                     : Neutral
  price_range_pct          : 0.2
  recent_patterns          : []
  nearest_support          : 1.15154
  support_strength         : Strong
  dist_to_support_pct      : 0.1
  nearest_resistance       : 1.15287
  resistance_strength      : Strong
  dist_to_resistance_pct   : 0.01
  htf_daily_bias           : Neutral
  htf_daily_structure      : Consolidation (Ranging)
  htf_1h_bias              : Neutral
  htf_1h_structure         : Consolidation (Ranging)
  daily_rsi                : 68.0
  daily_macd               : bearish
  daily_trend              : uptrend
  daily_support            : [np.float64(1.13545), np.float64(1.13788)]
  daily_resistance         : [np.float64(1.14705)]

In [7]:
# Analyse IA avec le contexte multi-timeframe
print("ANALYSE IA")
print("=" * 70)

ai_result = analyze_with_ai(signal)

if ai_result:
    signal_emoji = {"BUY": "🟢", "SELL": "🔴", "HOLD": "🟡"}
    ai_signal = ai_result.get('signal', 'UNKNOWN').upper()
    confidence = ai_result.get('confidence', 0)

    print(f"\nSignal:          {ai_signal} {signal_emoji.get(ai_signal, '⚪')}")
    print(f"Confiance:       {confidence}%")
    print(f"Raison:          {ai_result.get('reason', 'N/A')}")

    entry_val = ai_result.get('entry')
    sl_val = ai_result.get('stop_loss')
    tp_val = ai_result.get('take_profit')
    if entry_val is not None and sl_val is not None and tp_val is not None:
        try:
            entry_val, sl_val, tp_val = float(entry_val), float(sl_val), float(tp_val)
            print(f"Entrée:          {entry_val:.5f}")
            print(f"Stop Loss:       {sl_val:.5f}")
            print(f"Take Profit:     {tp_val:.5f}")
        except (ValueError, TypeError):
            print(f"Entrée:          {entry_val}")
            print(f"Stop Loss:       {sl_val}")
            print(f"Take Profit:     {tp_val}")
else:
    print("Analyse IA indisponible")


ANALYSE IA

Signal:          HOLD 🟡
Confiance:       65%
Raison:          The market is in consolidation (ranging) across multiple timeframes. While the daily trend is an uptrend and RSI is high (68), the immediate price action suggests a tight range between strong support (1.15154) and resistance (1.15287). Trading breakouts or reversals against the current tight range is too risky.
